# Chi-squared Manuscript Plots — Scenario D

Generates publication-quality figures for Chapter 3, Section 5.4 (Goodness-of-Fit Analysis).

**Scenario D**: Out-of-sample (excluding Kinney + Smith), combined uncertainties.

Plots produced:
1. Cumulative χ²/N vs. energy
2. Experiment-contribution waterfall
3. Per-experiment χ²/N vs. publication year
4. Standardized residual histograms

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'serif',
})

# ── Constants ─��────────────────────────────────────────────────────────
PARQUET_PATH = '/share_snc/snc/JuanMonleon/chi2/chi2_hybrid_data_new_test_35.parquet'
E_MIN, E_MAX = 0.85, 4.0
KINNEY_SMITH_IDS = ['10571002', '10886002']

LIB_ORDER  = ['This_work', 'JEFF', 'JENDL']
LIB_LABELS = {'JEFF': 'JEFF-4.0', 'JENDL': 'JENDL-5', 'This_work': 'This work'}
LIB_COLORS = {'JEFF': '#1f77b4', 'JENDL': '#ff7f0e', 'This_work': '#2ca02c'}
MARKERS    = {'JEFF': 's', 'JENDL': '^', 'This_work': 'o'}

N_ENERGY_BINS = 20

In [ ]:
# ── Load and filter to Scenario D ──────────────────────────────────
df_raw = pd.read_parquet(PARQUET_PATH)
df = df_raw[(df_raw['energy_mev'] >= E_MIN) & (df_raw['energy_mev'] <= E_MAX)].copy()

# Compute chi2 and residuals with combined uncertainty
df['chi2_total'] = df['residual_total'] ** 2
inf_mask = ~np.isfinite(df['chi2_total'])
if inf_mask.any():
    print(f"Dropping {inf_mask.sum()} non-finite chi2 rows")
    df = df[~inf_mask]

# Exclude Kinney + Smith
df_D = df[~df['experiment_id'].isin(KINNEY_SMITH_IDS)].copy()

# Keep only known libraries
df_D = df_D[df_D['library'].isin(LIB_ORDER)].copy()

print(f"Scenario D: {len(df_D)} rows, {df_D['experiment_id'].nunique()} experiments")
for lib in LIB_ORDER:
    sub = df_D[df_D['library'] == lib]
    n = len(sub)
    chi2_n = sub['chi2_total'].sum() / n if n > 0 else float('nan')
    print(f"  {LIB_LABELS[lib]:12s}  N={n:,}  chi2/N={chi2_n:.3f}")

In [ ]:
# ── Build per-experiment summary table ─────────────────────────────
exp_rows = []
for lib in LIB_ORDER:
    sub = df_D[df_D['library'] == lib]
    for eid, grp in sub.groupby('experiment_id'):
        n = len(grp)
        exp_rows.append({
            'library': lib,
            'experiment_id': eid,
            'author': grp['author'].iloc[0] if 'author' in grp.columns else '',
            'year': int(grp['year'].iloc[0]) if 'year' in grp.columns and pd.notna(grp['year'].iloc[0]) else None,
            'N': n,
            'chi2_total': grp['chi2_total'].sum(),
            'chi2/N': grp['chi2_total'].sum() / n if n > 0 else np.nan,
        })
exp_df = pd.DataFrame(exp_rows)
print(f"Per-experiment table: {len(exp_df)} rows")
exp_df.head()

## 1. Cumulative χ²/N vs. energy

In [ ]:
# ── Plot 1: Cumulative chi2/N vs energy ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))

for lib in LIB_ORDER:
    sub = df_D[df_D['library'] == lib].sort_values('energy_mev')
    cum_chi2 = sub['chi2_total'].cumsum().values
    cum_n = np.arange(1, len(sub) + 1)
    ax.plot(sub['energy_mev'].values, cum_chi2 / cum_n,
            color=LIB_COLORS[lib], label=LIB_LABELS[lib], linewidth=1.5)

ax.axhline(1.0, color='k', ls='--', lw=0.8, label=r'$\chi^2/N = 1$')
ax.set_xlabel('Energy (MeV)')
ax.set_ylabel(r'Cumulative $\chi^2/N$')
ax.legend(loc='upper right')
ax.set_xlim(E_MIN, E_MAX)
ax.set_ylim(bottom=0)
fig.tight_layout()
plt.show()

## 2. Experiment-contribution waterfall

In [ ]:
# ── Plot 2: Cumulative waterfall by experiment ─────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for lib in LIB_ORDER:
    sub = exp_df[exp_df['library'] == lib].sort_values('chi2_total', ascending=False).reset_index(drop=True)
    total = sub['chi2_total'].sum()
    sub['cum_pct'] = 100.0 * sub['chi2_total'].cumsum() / total
    ax.plot(range(1, len(sub) + 1), sub['cum_pct'], 'o-',
            color=LIB_COLORS[lib], label=LIB_LABELS[lib], markersize=5)

ax.axhline(80, color='#333333', ls='--', lw=2.0, alpha=0.9, label='80%')
ax.axhline(95, color='#333333', ls=':', lw=2.0, alpha=0.9, label='95%')
ax.set_xlabel(r'Experiment rank (by $\chi^2$ contribution)')
ax.set_ylabel(r'Cumulative $\chi^2$ (%)')
ax.legend()
fig.tight_layout()
plt.show()

## 3. Per-experiment χ²/N vs. publication year

In [ ]:
# ── Plot 3: chi2/N vs publication year ─────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

offsets = {'This_work': -0.3, 'JEFF': 0.0, 'JENDL': 0.3}

for lib in LIB_ORDER:
    sub = exp_df[exp_df['library'] == lib].copy()
    sub['year_num'] = pd.to_numeric(sub['year'], errors='coerce')
    sub = sub.dropna(subset=['year_num'])
    offset = offsets.get(lib, 0)
    ax.scatter(sub['year_num'] + offset, sub['chi2/N'],
               c=LIB_COLORS[lib], marker=MARKERS.get(lib, 'o'),
               s=50, alpha=0.7, edgecolors='k', linewidths=0.3,
               label=LIB_LABELS[lib], zorder=3)

ax.axvspan(1982, 2016, alpha=0.10, color='gray', zorder=0)
ax.axhline(1.0, color='k', ls='--', lw=0.8)
ax.set_xlabel('Publication year')
ax.set_ylabel(r'$\chi^2/N$')
ax.legend()
fig.tight_layout()
plt.show()

## 4. Standardized residual histograms

In [ ]:
# ── Plot 4: Residual histograms ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
x_grid = np.linspace(-8, 8, 300)

for ax, lib in zip(axes, LIB_ORDER):
    r = df_D.loc[df_D['library'] == lib, 'residual_total'].dropna().values
    mu_fit, sigma_fit = stats.norm.fit(r)
    ax.hist(r, bins=80, density=True, color=LIB_COLORS[lib],
            edgecolor='k', linewidth=0.3, alpha=0.7, label=LIB_LABELS[lib])
    ax.plot(x_grid, stats.norm.pdf(x_grid, 0, 1), 'k--', lw=2.0,
            label=r'$\mathcal{N}(0,1)$')
    ax.plot(x_grid, stats.norm.pdf(x_grid, mu_fit, sigma_fit), 'r-', lw=1.5,
            label=rf'Fit: $\mu$={mu_fit:.2f}, $\sigma$={sigma_fit:.2f}')
    ax.set_xlim(-8, 8)
    ax.set_xlabel('Standardized residual')
    ax.set_title(LIB_LABELS[lib])
    ax.legend(fontsize=9)

axes[0].set_ylabel('Probability density')
fig.tight_layout()
plt.show()